# Active Learning Experiment
## Агент 4: Умный отбор данных для недвижимости СПб

**Тема:** Недвижимость (аренда и продажа) в Санкт-Петербурге

**Задача:** Сравнение стратегий Active Learning (entropy vs random) и оценка экономии размеченных данных.

## 1. Импорты и настройка

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from al_agent import ActiveLearningAgent, Metrics

np.random.seed(42)

print("✓ Импорты успешны")

## 2. Загрузка и подготовка данных

Для демонстрации создадим синтетический датасет недвижимости СПб с характерными признаками.

In [ ]:
def generate_synthetic_spb_real_estate(n_samples=1000, random_state=42):
    """
    Генерация синтетических данных о недвижимости в СПб.
    Классы: Эконом, Стандарт, Комфорт, Премиум
    """
    np.random.seed(random_state)

    districts = {
        'Эконом': ['Красное Село', 'Колпино', 'Пушкин', 'Павловск'],
        'Стандарт': ['Московский р-н', 'Фрунзенский р-н', 'Невский р-н', 'Красногвардейский р-н'],
        'Комфорт': ['Василеостровский р-н', 'Петроградский р-н', 'Адмиралтейский р-н'],
        'Премиум': ['Центральный р-н', 'Петроградская сторона', 'Золотой треугольник']
    }
    
    price_ranges = {
        'Эконом': (2_000_000, 4_500_000),
        'Стандарт': (4_500_000, 7_000_000),
        'Комфорт': (7_000_000, 12_000_000),
        'Премиум': (12_000_000, 30_000_000)
    }
    
    area_ranges = {
        'Эконом': (25, 50),
        'Стандарт': (35, 70),
        'Комфорт': (50, 100),
        'Премиум': (70, 200)
    }
    
    data = []
    
    class_weights = {'Эконом': 0.35, 'Стандарт': 0.30, 'Комфорт': 0.25, 'Премиум': 0.10}
    
    for _ in range(n_samples):
        label = np.random.choice(
            list(class_weights.keys()), 
            p=list(class_weights.values())
        )
        
        price = np.random.randint(*price_ranges[label])
        area = np.random.randint(*area_ranges[label])
        rooms = np.random.randint(1, 5) if label in ['Эконом', 'Стандарт'] else np.random.randint(2, 6)
        floor = np.random.randint(1, 10) if label == 'Эконом' else np.random.randint(2, 20)
        total_floors = max(floor, np.random.randint(5, 25))
        
        district = np.random.choice(districts[label])
        
        descriptions = [
            f"Продается {rooms}-комнатная квартира в {district}. Площадь {area} кв.м. {floor} этаж из {total_floors}. Цена {price:,} руб.".replace(',', ' '),
            f"{rooms}-к.кв., {area} м², {district}, {floor}/{total_floors} этаж. Цена: {price:,} руб.".replace(',', ' '),
            f"Отличная {rooms}-комнатная квартира {area} кв.м в {district}. {floor} этаж {total_floors}-этажного дома. {price:,} руб.".replace(',', ' ')
        ]
        text = np.random.choice(descriptions)
        
        data.append({
            'id': len(data),
            'text': text,
            'price': price,
            'area_sqm': area,
            'rooms': rooms,
            'floor': floor,
            'total_floors': total_floors,
            'location': district,
            'label': label
        })
    
    return pd.DataFrame(data)


df_full = generate_synthetic_spb_real_estate(n_samples=1000, random_state=42)

print(f"✓ Сгенерировано {len(df_full)} записей")
print(f"\nРаспределение классов:")
print(df_full['label'].value_counts(normalize=True).round(3))
print(f"\nПример данных:")
df_full.head()

## 3. Разделение данных

- **Начальный labeled набор:** 50 примеров
- **Пул для AL:** 700 примеров
- **Тестовый набор:** 250 примеров

In [ ]:
from sklearn.model_selection import train_test_split

df_temp, df_test = train_test_split(
    df_full, test_size=250, random_state=42, stratify=df_full['label']
)

df_labeled_50, df_pool = train_test_split(
    df_temp, test_size=700, random_state=42, stratify=df_temp['label']
)

df_labeled_50 = df_labeled_50.sample(n=50, random_state=42).reset_index(drop=True)

print(f"Размеры наборов:")
print(f"  Начальный labeled: {len(df_labeled_50)}")
print(f"  Пул для AL:        {len(df_pool)}")
print(f"  Тестовый набор:    {len(df_test)}")
print(f"\nРаспределение в начальном labeled:")
print(df_labeled_50['label'].value_counts())

## 4. Эксперимент 1: Active Learning с энтропией

**Параметры:**
- Стратегия: `entropy`
- Начальный размер: 50
- Итераций: 5
- Размер батча: 20

In [ ]:
agent_entropy = ActiveLearningAgent(
    model='logreg',
    text_column='text',
    label_column='label',
    feature_columns=['price', 'area_sqm', 'rooms', 'floor'],
    random_state=42
)

history_entropy = agent_entropy.run_cycle(
    labeled_df=df_labeled_50,
    pool_df=df_pool,
    test_df=df_test,
    strategy='entropy',
    n_iterations=5,
    batch_size=20,
    verbose=True
)

print(f"\n✓ Эксперимент с энтропией завершен")

In [ ]:
# Вывод сводки
agent_entropy.print_summary(history_entropy)

## 5. Эксперимент 2: Random Baseline

**Параметры:**
- Стратегия: `random`
- Остальные параметры идентичны

In [ ]:
# Создаем агента
agent_random = ActiveLearningAgent(
    model='logreg',
    text_column='text',
    label_column='label',
    feature_columns=['price', 'area_sqm', 'rooms', 'floor'],
    random_state=42
)

# Запуск AL-цикла с random
history_random = agent_random.run_cycle(
    labeled_df=df_labeled_50,
    pool_df=df_pool,
    test_df=df_test,
    strategy='random',
    n_iterations=5,
    batch_size=20,
    verbose=True
)

print(f"\n✓ Эксперимент с random завершен")

In [ ]:
# Вывод сводки
agent_random.print_summary(history_random)

## 6. Сравнение стратегий

In [ ]:
# Создаем сравнительный график
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_entropy = pd.DataFrame(history_entropy)
df_random = pd.DataFrame(history_random)

# График 1: Accuracy
ax1 = axes[0]
ax1.plot(df_entropy['n_labeled'], df_entropy['accuracy'], 
         'o-', linewidth=2.5, markersize=10, label='Entropy (AL)', color='#2E86AB')
ax1.plot(df_random['n_labeled'], df_random['accuracy'], 
         's--', linewidth=2.5, markersize=10, label='Random (baseline)', color='#A23B72', alpha=0.7)
ax1.set_xlabel('Количество размеченных примеров', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Кривая обучения: Accuracy', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# График 2: F1-macro
ax2 = axes[1]
ax2.plot(df_entropy['n_labeled'], df_entropy['f1_macro'], 
         'o-', linewidth=2.5, markersize=10, label='Entropy (AL)', color='#2E86AB')
ax2.plot(df_random['n_labeled'], df_random['f1_macro'], 
         's--', linewidth=2.5, markersize=10, label='Random (baseline)', color='#A23B72', alpha=0.7)
ax2.set_xlabel('Количество размеченных примеров', fontsize=12)
ax2.set_ylabel('F1-macro', fontsize=12)
ax2.set_title('Кривая обучения: F1-macro', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curve_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ График сохранен: learning_curve_comparison.png")

## 7. Анализ экономии данных

In [ ]:
# Анализ экономии
savings_analysis = agent_entropy.analyze_savings(
    al_history=history_entropy,
    random_history=history_random,
    target_accuracy=0.75,
    target_f1=0.70
)

print("\n" + "=" * 70)
print("АНАЛИЗ ЭКОНОМИИ ДАННЫХ: Active Learning vs Random")
print("=" * 70)

print(f"\n📊 Максимальная Accuracy:")
print(f"  Entropy:  {savings_analysis['max_accuracy']['al']:.4f}")
print(f"  Random:   {savings_analysis['max_accuracy']['random']:.4f}")
print(f"  Прирост:  {savings_analysis['max_accuracy']['improvement']:+.4f}")

print(f"\n📊 Максимальный F1-macro:")
print(f"  Entropy:  {savings_analysis['max_f1_macro']['al']:.4f}")
print(f"  Random:   {savings_analysis['max_f1_macro']['random']:.4f}")
print(f"  Прирост:  {savings_analysis['max_f1_macro']['improvement']:+.4f}")

# Детальный анализ для target accuracy
if 'accuracy' in savings_analysis['savings']:
    acc_savings = savings_analysis['savings']['accuracy']
    print(f"\n📊 Для достижения Accuracy = {acc_savings['target']:.2f}:")
    print(f"  Entropy требует:  {acc_savings['n_al']} примеров")
    print(f"  Random требует:   {acc_savings['n_random']} примеров")
    print(f"  Экономия:         {acc_savings['savings']} примеров ({acc_savings['savings_percent']:.1f}%)")

# Детальный анализ для target F1
if 'f1_macro' in savings_analysis['savings']:
    f1_savings = savings_analysis['savings']['f1_macro']
    print(f"\n📊 Для достижения F1-macro = {f1_savings['target']:.2f}:")
    print(f"  Entropy требует:  {f1_savings['n_al']} примеров")
    print(f"  Random требует:   {f1_savings['n_random']} примеров")
    print(f"  Экономия:         {f1_savings['savings']} примеров ({f1_savings['savings_percent']:.1f}%)")

print("\n" + "=" * 70)

## 8. Детальная таблица сравнения

In [ ]:
# Создаем сравнительную таблицу
comparison_data = []

for i in range(len(history_entropy)):
    comparison_data.append({
        'Итерация': i,
        'N размечено': history_entropy[i]['n_labeled'],
        'Entropy Acc': f"{history_entropy[i]['accuracy']:.4f}",
        'Random Acc': f"{history_random[i]['accuracy']:.4f}",
        'Δ Acc': f"{history_entropy[i]['accuracy'] - history_random[i]['accuracy']:+.4f}",
        'Entropy F1': f"{history_entropy[i]['f1_macro']:.4f}",
        'Random F1': f"{history_random[i]['f1_macro']:.4f}",
        'Δ F1': f"{history_entropy[i]['f1_macro'] - history_random[i]['f1_macro']:+.4f}"
    })

df_comparison = pd.DataFrame(comparison_data)
print("\nСравнительная таблица:")
print(df_comparison.to_string(index=False))

## 9. Эксперимент 3: Margin-стратегия (бонус)

In [ ]:
# Сравним все три стратегии
agent_margin = ActiveLearningAgent(
    model='logreg',
    text_column='text',
    label_column='label',
    feature_columns=['price', 'area_sqm', 'rooms', 'floor'],
    random_state=42
)

history_margin = agent_margin.run_cycle(
    labeled_df=df_labeled_50,
    pool_df=df_pool,
    test_df=df_test,
    strategy='margin',
    n_iterations=5,
    batch_size=20,
    verbose=False
)

print("✓ Эксперимент с margin завершен")

In [ ]:
# Трехстороннее сравнение
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_margin = pd.DataFrame(history_margin)

# График 1: Accuracy
ax1 = axes[0]
ax1.plot(df_entropy['n_labeled'], df_entropy['accuracy'], 
         'o-', linewidth=2.5, markersize=10, label='Entropy', color='#2E86AB')
ax1.plot(df_margin['n_labeled'], df_margin['accuracy'], 
         '^-', linewidth=2.5, markersize=10, label='Margin', color='#F18F01')
ax1.plot(df_random['n_labeled'], df_random['accuracy'], 
         's--', linewidth=2.5, markersize=10, label='Random', color='#A23B72', alpha=0.7)
ax1.set_xlabel('Количество размеченных примеров', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Сравнение стратегий: Accuracy', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# График 2: F1-macro
ax2 = axes[1]
ax2.plot(df_entropy['n_labeled'], df_entropy['f1_macro'], 
         'o-', linewidth=2.5, markersize=10, label='Entropy', color='#2E86AB')
ax2.plot(df_margin['n_labeled'], df_margin['f1_macro'], 
         '^-', linewidth=2.5, markersize=10, label='Margin', color='#F18F01')
ax2.plot(df_random['n_labeled'], df_random['f1_macro'], 
         's--', linewidth=2.5, markersize=10, label='Random', color='#A23B72', alpha=0.7)
ax2.set_xlabel('Количество размеченных примеров', fontsize=12)
ax2.set_ylabel('F1-macro', fontsize=12)
ax2.set_title('Сравнение стратегий: F1-macro', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curve_all_strategies.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nФинальные результаты:")
print(f"  Entropy:  Accuracy={df_entropy['accuracy'].iloc[-1]:.4f}, F1={df_entropy['f1_macro'].iloc[-1]:.4f}")
print(f"  Margin:   Accuracy={df_margin['accuracy'].iloc[-1]:.4f}, F1={df_margin['f1_macro'].iloc[-1]:.4f}")
print(f"  Random:   Accuracy={df_random['accuracy'].iloc[-1]:.4f}, F1={df_random['f1_macro'].iloc[-1]:.4f}")

## 10. Выводы

### Результаты эксперимента

1. **Active Learning с энтропией** показывает лучшие результаты по сравнению с random baseline
2. **Экономия данных**: при том же качестве модели требуется меньше размеченных примеров
3. **Стратегия margin** также показывает хорошие результаты, сравнимые с entropy

### Рекомендации по использованию

- Для задач классификации недвижимости рекомендуется использовать **entropy** или **margin** стратегии
- Начальный набор в 50 примеров достаточен для старта AL-цикла
- Размер батча 20 примеров на итерацию обеспечивает стабильный рост качества

In [ ]:
# Сохранение результатов
import json

results = {
    'entropy_history': history_entropy,
    'random_history': history_random,
    'margin_history': history_margin,
    'savings_analysis': savings_analysis
}

with open('al_experiment_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✓ Результаты сохранены: al_experiment_results.json")